Silver to Gold  table

Based on a EDA done in R we slescted predictors by doing a  AIC and BIC.
The AIC and BIC stepwise logistic regression models produced very similar predictive performance. The BIC model had slightly better accuracy, Gamma sensitivity, and Hadron specificity, while the AIC model had a very slightly higher AUC. Since the AUC difference was negligible and BIC favors a simpler model, the BIC model was selected for the Gold layer and final MLflow pipeline.

The Gold table will contain the BIC-selected predictors and the binary target variable:

- `f_length`
- `f_size`
- `f_conc1`
- `f_m3long`
- `f_alpha`
- `class_binary`

In [0]:
# ============================================================
# SILVER TO GOLD
# GAMMA TELESCOPE
# PySpark ML-ready table using BIC-selected features
# ============================================================

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# ============================================================
# 1. Load Silver table
# ============================================================

silver_table_name = "workspace.medallion_data.silver_telescope"
gold_table_name = "workspace.medallion_data.gold_telescope"

silver_df = spark.table(silver_table_name)

display(silver_df.limit(10))
silver_df.printSchema()

print("Silver row count:", silver_df.count())
print("Silver column count:", len(silver_df.columns))

In [0]:
# ============================================================
# 2. Define BIC-selected features
# ============================================================

bic_features = [
    "f_length",
    "f_size",
    "f_conc1",
    "f_m3long",
    "f_alpha"
]

print("Features used for Gold table:")
print(bic_features)

In [0]:
# ============================================================
# 3. Select features and target class
# ============================================================

gold_base_df = (
    silver_df
    .select(
        *bic_features,
        "class"
    )
    .dropna()
)

display(gold_base_df.limit(10))

print("Class distribution:")
display(gold_base_df.groupBy("class").count())

In [0]:
# ============================================================
# 7. Save Gold table
# Pure Medallion architecture: Clean columns, no VectorAssembler
# ============================================================

(
    gold_base_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_name)
)

print(f"Gold table saved as: {gold_table_name}")

In [0]:
# ============================================================
# 8. Check saved Gold table
# ============================================================

gold_check = spark.table(gold_table_name)

display(gold_check.limit(10))
gold_check.printSchema()

gold_rows = gold_check.count()
gold_cols = len(gold_check.columns)

print("Gold table shape:")
print(f"Rows: {gold_rows}")
print(f"Columns: {gold_cols}")
print(f"Shape: ({gold_rows}, {gold_cols})")

In [0]:
# Final medallion table check
for table_name in [
    "workspace.medallion_data.bronze_telescope",
    "workspace.medallion_data.silver_telescope",
    "workspace.medallion_data.gold_telescope"
]:
    df = spark.table(table_name)
    print(f"{table_name}")
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    display(df.limit(5))